# 1. Import and Install Dependencies

In [1]:
!pip install tensorflow==2.5.1 tensorflow-gpu==2.5.1 opencv-python mediapipe sklearn matplotlib

ERROR: Could not find a version that satisfies the requirement tensorflow==2.5.1 (from versions: 2.8.0rc1, 2.8.0, 2.8.1, 2.8.2, 2.8.3, 2.8.4, 2.9.0rc0, 2.9.0rc1, 2.9.0rc2, 2.9.0, 2.9.1, 2.9.2, 2.9.3, 2.10.0rc0, 2.10.0rc1, 2.10.0rc2, 2.10.0rc3, 2.10.0, 2.10.1, 2.11.0rc0, 2.11.0rc1, 2.11.0rc2, 2.11.0, 2.12.0rc0)
ERROR: No matching distribution found for tensorflow==2.5.1


In [1]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import datetime
import mediapipe as mp
import datetime

# 2. Keypoints using MP Holistic

In [3]:
def GetFileName():
        x = datetime.datetime.now()
        s = x.strftime('%Y-%m-%d-%H%M%S%f')
        return s
def CreateDir(path):
    ls = [];
    head_tail = os.path.split(path)
    ls.append(path)
    while len(head_tail[1])>0:
        head_tail = os.path.split(path)
        path = head_tail[0]
        ls.append(path)
        head_tail = os.path.split(path)   
    for i in range(len(ls)-2,-1,-1):
        sf =ls[i]
        isExist = os.path.exists(sf)
        if not isExist:
            os.makedirs(sf)
NamaDataSet = "tes"

In [4]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [5]:
def draw_landmarks(image, results):
    #mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION) # Draw face connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(bimage, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
 

In [6]:
def draw_styled_landmarks(image, results):
    # Draw face connections
   # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
                            # mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), 
                            # mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
                            # ) 
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 

In [6]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    
    image_height, image_width, _ = image.shape
    coords = []
    coordsy = []
    
    
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR    
  
    if results.pose_landmarks:
        pose_landmarks = results.pose_landmarks.landmark
        for A in pose_landmarks:
          cx, cy = A.x * image_width, A.y*image_height                                                                                                                                                                                                                                                                                                                                                                                            
          coords.append(cx) 
          coordsy.append(cy) 
        
        x_max = max(coords)
        y_max = max(coordsy)
        x_min = min(coords)
        y_min = min(coordsy)

        cv2.rectangle(image, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (0, 255, 0), 2)
    
    return image, results

In [8]:
cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        

        # Show to screen
        cv2.imshow('MediaPipe Pose', cv2.flip(image, 1))
        if cv2.waitKey(10) & 0xFF == ord('q'):
          break
    cap.release()
    cv2.destroyAllWindows()

In [7]:
def Dataset(NoKamera,NamaDataSet):
    DirektoriData = "c:\\temp\\dataimage"+"\\"+NamaDataSet+"\\"+GetFileName()    
    CreateDir(DirektoriData)        
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles
    mp_pose = mp.solutions.pose
    imsize=(640, 480)
    TimeStart = time.time() 
    TimeNow = time.time() +10
    FrameRate = 5
    
    
    cap = cv2.VideoCapture(NoKamera,cv2.CAP_DSHOW)
    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
      
      while cap.isOpened():
        success, image = cap.read()
        if not success:
          print("Ignoring empty camera frame.")
          # If loading a video, use 'break' instead of 'continue'.
          continue
    

        image.flags.writeable = False                  # Image is no longer writeable
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, imsize)
        results = holistic.process(image)                 # Make prediction
        
        image_height, image_width, _ = image.shape
        coords = []
        coordsy = []

        if results.pose_landmarks:
            pose_landmarks = results.pose_landmarks.landmark
            for A in pose_landmarks:
              cx, cy = A.x * image_width, A.y*image_height                                                                                                                                                                                                                                                                                                                                                                                            
              coords.append(cx) 
              coordsy.append(cy) 

            x_max = max(coords)
            y_max = max(coordsy)
            x_min = min(coords)
            y_min = min(coordsy)
            

            cv2.rectangle(image, (int(x_min), int(y_min)), (int(x_max), int(y_max)), (255, 255, 0), 2)

    
        image.flags.writeable = True                   # Image is now writeable 
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
        bimage = np.zeros((image_height,image_width,3), np.uint8)
        cv2.rectangle(bimage,(int(x_min), int(y_min)),(int(x_max), int(y_max)),(0,255,0),2)
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
        mp_drawing.draw_landmarks(bimage, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
        if (int (x_max) < 0):
            x_max = 1
        elif (int (x_min) < 0):
            x_min = 1
        elif (int (x_max) < 0):
            y_max = 1
        elif (int (x_min) < 0):
            y_min = 1    
              

        image = cv2.rectangle(image,(int(x_min), int(y_min)),(int(x_max), int(y_max)),(255,255,0),2)
        image = cv2.cvtColor(black, cv2.COLOR_RGB2BGR)
 
        cropped_image = bimage[(int(y_min)):(int(y_max)), (int(x_min)):(int(x_max)),:]
        dy =y_max -y_min
        dx = x_max -x_min
        print(dy,dx)
        print(cropped_image.shape) 
        TimeNow = time.time() 
        if TimeNow-TimeStart>1/FrameRate:
            print(cropped_image.shape)
            TimeStart = TimeNow
            sFile = DirektoriData+"\\"+GetFileName()
            imsize2=(128,128)
            cropped_image = cv2.resize(cropped_image, imsize2)
            cv2.imwrite(sFile+'.jpg', cropped_image)
            cv2.imwrite(sFile+'.png', image)
        
        cv2.imshow('MediaPipe Pose', cv2.flip(image, 1))
        if cv2.waitKey(10) & 0xFF == ord('q'):
              break
    cap.release()
    cv2.destroyAllWindows()

In [8]:
Dataset(0,"HurufAZ")

NameError: name 'black' is not defined

: 

In [11]:
cap.release()
cv2.destroyAllWindows()